# NISQA-SIM Low-MOS Mix Generator (Active-Region, Single Distortion)

This notebook creates a low-MOS mixed dataset where each file has exactly one distortion segment.

Design goals:
- Pick source pairs from low-MOS rows only
- Insert exactly one DEG segment per file
- Place segment start in active speech (non-silent) regions
- Keep distortion windows inside high-activity regions as much as possible

Segment placement now uses a median-like energy criterion (window energy >= median quantile).


In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display
from scipy.signal import resample_poly

In [ ]:
@dataclass
class Segment:
    """Represents a time interval in seconds."""

    start: float
    end: float


DATA_ROOT = Path("../data/raw/NISQA_Corpus")
SIM_SPLIT = "NISQA_TRAIN_SIM"
CSV_PATH = DATA_ROOT / SIM_SPLIT / f"{SIM_SPLIT}_file.csv"
OUTPUT_DIR = Path("../data/processed/nisqa_sim_mix_lowmos_active_40")
MANIFEST_PATH = OUTPUT_DIR / "manifest.csv"

TOTAL_MIX_FILES = 40
MOS_MAX_THRESHOLD = 2.2
REQUIRE_ACTIVE_DEGRADATION_TYPES = True

TARGET_SAMPLE_RATE = 16000
MAX_DURATION_SECONDS = None
SEED = 42

SEGMENT_MIN_SECONDS = 1.0
SEGMENT_MAX_RATIO = 0.40
MIN_ACTIVE_FRACTION = 0.55
ACTIVITY_STD_MULTIPLIER = 0.65
ACTIVITY_ABS_FLOOR = 0.006
ACTIVITY_SMOOTH_MS = 20
SEGMENT_ATTEMPTS = 300
ENERGY_QUANTILE = 0.50
ENERGY_TOP_K_RATIO = 0.25
OUTPUT_ACTIVE_FRACTION_MIN = 0.50

PLOT_EXAMPLES = 10

DEGRADATION_COLUMNS = [
    "filter",
    "timeclipping",
    "wbgn",
    "p50mnru",
    "bgn",
    "clipping",
    "arb_filter",
    "codec1",
    "codec2",
    "codec3",
    "plcMode1",
    "plcMode2",
    "plcMode3",
]

rng = random.Random(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_audio_mono(path: Path) -> tuple[np.ndarray, int]:
    """Load waveform and return mono float32 samples plus sample rate."""

    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    return audio.astype(np.float32), int(sr)


def resample_if_needed(audio: np.ndarray, sr_in: int, sr_out: int) -> np.ndarray:
    """Resample only when input sample rate differs from target sample rate."""

    if sr_in == sr_out:
        return audio
    gcd = math.gcd(sr_in, sr_out)
    up = sr_out // gcd
    down = sr_in // gcd
    return resample_poly(audio, up=up, down=down).astype(np.float32)


def align_pair(ref_audio: np.ndarray, deg_audio: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Align REF/DEG waveforms to a shared length without padding."""

    length = min(len(ref_audio), len(deg_audio))
    if MAX_DURATION_SECONDS is not None:
        max_len = int(round(MAX_DURATION_SECONDS * TARGET_SAMPLE_RATE))
        length = min(length, max_len)
    return ref_audio[:length], deg_audio[:length]


def is_active_tag(value: object) -> bool:
    """Return True if metadata cell indicates an active degradation tag."""

    if pd.isna(value):
        return False
    token = str(value).strip()
    return token not in {"", "-", "nan", "None"}


def extract_active_degradations(row: pd.Series) -> list[str]:
    """Extract active source degradation tags from configured metadata columns."""

    return [col for col in DEGRADATION_COLUMNS if is_active_tag(row.get(col, np.nan))]


def smooth_envelope(audio: np.ndarray, sr: int, smooth_ms: int) -> np.ndarray:
    """Compute smoothed absolute-amplitude envelope."""

    win = max(1, int(round(sr * (smooth_ms / 1000.0))))
    kernel = np.ones(win, dtype=np.float32) / float(win)
    return np.convolve(np.abs(audio), kernel, mode="same").astype(np.float32)


def choose_active_segment(ref_audio: np.ndarray, sr: int) -> tuple[Segment, float, float]:
    """Choose one segment in active audio using high-energy window ranking."""

    n = len(ref_audio)
    total_seconds = n / float(sr)
    env = smooth_envelope(ref_audio, sr, ACTIVITY_SMOOTH_MS)
    threshold = max(ACTIVITY_ABS_FLOOR, ACTIVITY_STD_MULTIPLIER * float(np.std(ref_audio)))
    max_seg_seconds = max(SEGMENT_MIN_SECONDS, total_seconds * SEGMENT_MAX_RATIO)

    def sample_seg_samples() -> int:
        seg_seconds = rng.uniform(SEGMENT_MIN_SECONDS, max_seg_seconds)
        seg_samples = max(1, int(round(seg_seconds * sr)))
        return min(seg_samples, max(1, n - 1))

    def window_energy_sums(signal: np.ndarray, win: int) -> np.ndarray:
        if win <= 1:
            return signal.copy()
        if len(signal) < win:
            return np.array([], dtype=np.float32)
        csum = np.concatenate(([0.0], np.cumsum(signal, dtype=np.float64)))
        sums = csum[win:] - csum[:-win]
        return sums.astype(np.float32)

    for _ in range(SEGMENT_ATTEMPTS):
        seg_samples = sample_seg_samples()
        window_energy = window_energy_sums(env, seg_samples)
        if len(window_energy) == 0:
            continue

        valid_len = len(window_energy)
        energy_cutoff = float(np.quantile(window_energy, ENERGY_QUANTILE))

        starts = np.arange(valid_len)
        candidates = starts[(window_energy >= energy_cutoff) & (env[:valid_len] >= threshold)]
        if len(candidates) == 0:
            continue

        candidate_energy = window_energy[candidates]
        top_k = max(1, int(math.ceil(len(candidates) * ENERGY_TOP_K_RATIO)))
        top_indices = np.argsort(candidate_energy)[-top_k:]
        pool = candidates[top_indices]

        start_idx = int(pool[rng.randrange(len(pool))])
        end_idx = min(n, start_idx + seg_samples)
        if end_idx <= start_idx:
            continue

        active_frac = float(np.mean(env[start_idx:end_idx] >= threshold))
        if active_frac >= MIN_ACTIVE_FRACTION:
            return Segment(start_idx / sr, end_idx / sr), threshold, active_frac

    fallback_seg = max(1, int(round(max(SEGMENT_MIN_SECONDS, min(max_seg_seconds, total_seconds * 0.33)) * sr)))
    fallback_seg = min(fallback_seg, max(1, n - 1))

    window_energy = window_energy_sums(env, fallback_seg)
    if len(window_energy) == 0:
        return Segment(0.0, min(total_seconds, SEGMENT_MIN_SECONDS)), threshold, 0.0

    valid_len = len(window_energy)
    starts = np.arange(valid_len)
    start_mask = env[:valid_len] >= threshold

    if np.any(start_mask):
        weighted = np.where(start_mask, window_energy, -np.inf)
        best_start = int(np.argmax(weighted))
    else:
        best_start = int(np.argmax(window_energy))

    start_idx = best_start
    end_idx = min(n, start_idx + fallback_seg)
    active_frac = float(np.mean(env[start_idx:end_idx] >= threshold)) if end_idx > start_idx else 0.0
    return Segment(start_idx / sr, end_idx / sr), threshold, active_frac


def build_mix_one_segment(
    ref_audio: np.ndarray,
    deg_audio: np.ndarray,
    sr: int,
) -> tuple[np.ndarray, Segment, float, float]:
    """Create mix with exactly one active-region DEG segment."""

    segment, threshold, active_frac = choose_active_segment(ref_audio, sr)

    mixed = ref_audio.copy()
    i0 = max(0, min(int(round(segment.start * sr)), len(mixed)))
    i1 = max(i0, min(int(round(segment.end * sr)), len(mixed)))
    mixed[i0:i1] = deg_audio[i0:i1]
    return mixed, segment, threshold, active_frac


def serialize_segments(segments: list[Segment]) -> str:
    """Serialize segment list as JSON string."""

    payload = [{"start": round(s.start, 3), "end": round(s.end, 3)} for s in segments]
    return json.dumps(payload)


def timeline_from_segment(duration_seconds: float, segment: Segment) -> list[dict]:
    """Build REF/DEG timeline from one DEG segment."""

    timeline: list[dict] = []

    if segment.start > 0.0:
        timeline.append({"start": 0.0, "end": round(segment.start, 3), "source": "REF"})

    timeline.append({"start": round(segment.start, 3), "end": round(segment.end, 3), "source": "DEG"})

    if segment.end < duration_seconds:
        timeline.append({"start": round(segment.end, 3), "end": round(duration_seconds, 3), "source": "REF"})

    return timeline


def switch_points_from_timeline(timeline: list[dict]) -> list[float]:
    """Return sorted unique boundary points from timeline entries."""

    pts: list[float] = []
    for item in timeline:
        pts.extend([float(item["start"]), float(item["end"])])
    return sorted(set(round(p, 3) for p in pts))

In [ ]:
df = pd.read_csv(CSV_PATH)
df["ref_path"] = df["filepath_ref"].apply(lambda p: DATA_ROOT / p)
df["deg_path"] = df["filepath_deg"].apply(lambda p: DATA_ROOT / p)

df = df[df["ref_path"].apply(Path.exists) & df["deg_path"].apply(Path.exists)].copy()
df = df[df["mos"].notna() & (df["mos"] <= MOS_MAX_THRESHOLD)].copy()
df = df.reset_index(drop=True)

df["active_degradation_types"] = df.apply(extract_active_degradations, axis=1)
df["num_source_degradation_types"] = df["active_degradation_types"].apply(len)

if REQUIRE_ACTIVE_DEGRADATION_TYPES:
    df = df[df["num_source_degradation_types"] > 0].copy().reset_index(drop=True)

if len(df) < TOTAL_MIX_FILES:
    raise ValueError(
        f"Need at least {TOTAL_MIX_FILES} low-MOS rows after filtering, found {len(df)}. "
        "Try increasing MOS_MAX_THRESHOLD."
    )

print(f"Eligible pool size: {len(df)} rows from {SIM_SPLIT} with MOS <= {MOS_MAX_THRESHOLD}.")
print("Rows with active degradation tags:", int((df["num_source_degradation_types"] > 0).sum()))

active_tag_counts = df["active_degradation_types"].explode().value_counts()
if len(active_tag_counts) > 0:
    display(active_tag_counts.rename("count").to_frame())

display(
    df[[
        "filename_deg",
        "mos",
        "active_degradation_types",
        "num_source_degradation_types",
    ]].sort_values("mos").head(20).reset_index(drop=True)
)


In [ ]:
records: list[dict] = []

source_indices = list(df.index)
rng.shuffle(source_indices)

for src_idx in source_indices:
    if len(records) >= TOTAL_MIX_FILES:
        break

    row = df.loc[src_idx]

    ref_audio, ref_sr = load_audio_mono(row["ref_path"])
    deg_audio, deg_sr = load_audio_mono(row["deg_path"])

    ref_audio = resample_if_needed(ref_audio, ref_sr, TARGET_SAMPLE_RATE)
    deg_audio = resample_if_needed(deg_audio, deg_sr, TARGET_SAMPLE_RATE)
    ref_audio, deg_audio = align_pair(ref_audio, deg_audio)

    mixed_audio, segment, threshold, active_frac = build_mix_one_segment(
        ref_audio=ref_audio,
        deg_audio=deg_audio,
        sr=TARGET_SAMPLE_RATE,
    )

    if active_frac < OUTPUT_ACTIVE_FRACTION_MIN:
        continue

    idx = len(records)
    stem = Path(row["filename_deg"]).stem
    out_path = OUTPUT_DIR / f"{idx:03d}_mix_{stem}.wav"
    sf.write(out_path, mixed_audio, TARGET_SAMPLE_RATE)

    duration_seconds = round(len(mixed_audio) / TARGET_SAMPLE_RATE, 3)
    text_segments = json.dumps([{"start": 0.0, "end": duration_seconds}])
    timeline = timeline_from_segment(duration_seconds, segment)
    switch_points = switch_points_from_timeline(timeline)

    records.append({
        "index": idx,
        "filename_ref": row["filename_ref"],
        "filename_deg": row["filename_deg"],
        "mos": float(row["mos"]),
        "duration_seconds": duration_seconds,
        "text_segments": text_segments,
        "mix_deg_segments": serialize_segments([segment]),
        "switch_points": json.dumps(switch_points),
        "mix_timeline": json.dumps(timeline),
        "source_degradation_types": json.dumps(row["active_degradation_types"]),
        "num_source_degradation_types": int(row["num_source_degradation_types"]),
        "activity_threshold": round(float(threshold), 6),
        "segment_active_fraction": round(float(active_frac), 4),
    })

if len(records) < TOTAL_MIX_FILES:
    raise ValueError(
        f"Generated only {len(records)} files with active fraction >= {OUTPUT_ACTIVE_FRACTION_MIN}. "
        "Increase pool size or relax threshold."
    )

manifest_df = pd.DataFrame(records).sort_values("index").reset_index(drop=True)
manifest_df.to_csv(MANIFEST_PATH, index=False)

print(f"Wrote {len(manifest_df)} mixed files to {OUTPUT_DIR}")
print(f"Manifest: {MANIFEST_PATH}")

display(manifest_df[["mos", "duration_seconds", "segment_active_fraction"]].describe())
display(
    manifest_df[[
        "index",
        "filename_deg",
        "mos",
        "mix_deg_segments",
        "source_degradation_types",
        "segment_active_fraction",
    ]]
)


In [ ]:
table_df = manifest_df[[
    "index",
    "filename_deg",
    "mos",
    "mix_deg_segments",
    "source_degradation_types",
    "segment_active_fraction",
]].sort_values("index").reset_index(drop=True)

display(table_df)

plot_df = manifest_df.sort_values("index").head(min(PLOT_EXAMPLES, len(manifest_df)))

for _, row in plot_df.iterrows():
    print()
    stem = Path(row["filename_deg"]).stem
    mix_path = OUTPUT_DIR / f"{int(row['index']):03d}_mix_{stem}.wav"

    mix_audio, mix_sr = load_audio_mono(mix_path)
    mix_audio = resample_if_needed(mix_audio, mix_sr, TARGET_SAMPLE_RATE)

    deg_segments = json.loads(row["mix_deg_segments"])
    active_types = json.loads(row["source_degradation_types"])

    times = np.arange(len(mix_audio)) / TARGET_SAMPLE_RATE
    fig, ax = plt.subplots(figsize=(12, 3.2))

    for seg in deg_segments:
        ax.axvspan(seg["start"], seg["end"], alpha=0.25, color="#ffb347")

    ax.plot(times, mix_audio, linewidth=0.65, color="#111111")
    ax.set_title("MIX waveform (orange background = DEG region)")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")

    types_label = ", ".join(active_types) if active_types else "none"
    fig.suptitle(
        f"Example {int(row['index']):03d} | MOS={row['mos']:.2f} | active={types_label}",
        y=1.04,
    )
    plt.tight_layout()
    plt.show()

    print(f"mix_deg_segments={row['mix_deg_segments']}")
    print(f"segment_active_fraction={row['segment_active_fraction']}")
    display(Audio(filename=str(mix_path)))

## Output

- Audio files: `../data/processed/nisqa_sim_mix_lowmos_active_40/*.wav`
- Manifest: `../data/processed/nisqa_sim_mix_lowmos_active_40/manifest.csv`

Manifest highlights:
- `mix_deg_segments`: exactly one DEG segment per file
- `source_degradation_types`: active NISQA degradation tags
- `segment_active_fraction`: active-audio coverage in the segment
- `activity_threshold`: per-file threshold used for active-region selection

Selection details:
- Window-energy criterion: candidates are above `ENERGY_QUANTILE` (default median, 0.50).
- Single segment is longer by default (`SEGMENT_MIN_SECONDS=1.0`, up to `SEGMENT_MAX_RATIO=0.40`).
